In [1]:
import torch
from unsloth import FastLanguageModel
import src.config as config
from src.train.lora_train import train_model
from src.train.sgd_full_train import train_model_sgd
from src.train.train_pipeline import train_pipeline
from src.data.prompt import mbpp
from src.data.prompt import openthoughts
from src.data.prompt import pythoncodes
from huggingface_hub import snapshot_download

import gc

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
WARNING 02-21 19:39:06 [interface.py:409] Using 'pin_memory=False' as WSL is detected. This may slow down the performance.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [2]:
snapshot_download(
    repo_id=config.MODEL_NAME,
    local_dir=config.MODEL_PATH,
    local_dir_use_symlinks=False,
    resume_download=True
)

'/workspace/models/qwen3-0.6b'

In [3]:
def run_experiments(experiments, skip_n):
	for i, (experiment_name, use_lora, exp) in enumerate(experiments):
		if skip_n > 0:
			skip_n -= 1
			continue

		print(f"=== Experiment {i+1}/{len(experiments)}: {experiment_name} ===")

		gc.collect()
		torch.cuda.empty_cache()

		path = config.MODEL_PATH + ("-sft" if use_lora == "lora" else "-sgd") + f"-{i+1}-exp"
		model, tokenizer = train_pipeline(exp, use_lora=use_lora, save_path=path)
	return model, tokenizer

In [4]:
experiments = [
	(
		"Double Training (LoRA)", "lora",
		[
			("Openthoughts-114k", openthoughts.get_prepared_dataset, {
				"lr": 2e-4,
				"batch_size": 4,
				"gradient_steps": 4,
				"max_steps": 500,
				"warmup_steps": 25,
				"optim": "paged_adamw_8bit",
				"assistant_only_loss": True,
			}),
			("Pythoncodes", pythoncodes.get_prepared_dataset, {
				"lr": 2e-5,
				"batch_size": 8,
				"gradient_steps": 8,
				"max_steps": 500,
				"warmup_steps": 25,
				"optim": "paged_adamw_8bit",
				"assistant_only_loss": True,
			}),
		]
	),
	(
		"Double Training (Full Train)", "full_train",
		[
			("Openthoughts-114k", openthoughts.get_prepared_dataset, {
				"lr": 3e-6,
				"batch_size": 4,
				"gradient_steps": 4,
				"max_steps": 500,
				"warmup_steps": 25,
				"optim": "paged_adamw_8bit",
				"assistant_only_loss": True,
			}),
			("Pythoncodes", pythoncodes.get_prepared_dataset, {
				"lr": 4e-7,
				"batch_size": 8,
				"gradient_steps": 8,
				"max_steps": 500,
				"warmup_steps": 25,
				"optim": "paged_adamw_8bit",
				"assistant_only_loss": True,
			}),
		]
	),
	(
		"Inverse Double Training (LoRA)", "lora",
		[
			("Pythoncodes", pythoncodes.get_prepared_dataset, {
				"lr": 1e-4,
				"batch_size": 8,
				"gradient_steps": 8,
				"max_steps": 500,
				"warmup_steps": 25,
				"optim": "paged_adamw_8bit",
				"assistant_only_loss": True,
			}),
			("Openthoughts-114k", openthoughts.get_prepared_dataset, {
				"lr": 3e-5,
				"batch_size": 4,
				"gradient_steps": 4,
				"max_steps": 500,
				"warmup_steps": 25,
				"optim": "paged_adamw_8bit",
				"assistant_only_loss": True,
			}),
		]
	),
	(
		"Inverse Double Training (Full Train)", "full_train",
		[
			("Pythoncodes", pythoncodes.get_prepared_dataset, {
				"lr": 1e-5,
				"batch_size": 8,
				"gradient_steps": 8,
				"max_steps": 500,
				"warmup_steps": 25,
				"optim": "paged_adamw_8bit",
				"assistant_only_loss": True,
			}),
			("Openthoughts-114k", openthoughts.get_prepared_dataset, {
				"lr": 3e-7,
				"batch_size": 4,
				"gradient_steps": 4,
				"max_steps": 500,
				"warmup_steps": 25,
				"optim": "paged_adamw_8bit",
				"assistant_only_loss": True,
			}),
		]
	),
	(
		"OpenThoughts (LoRA)", "lora",
		[
			("Openthoughts-114k", openthoughts.get_prepared_dataset, {
				"lr": 3e-6,
				"batch_size": 4,
				"gradient_steps": 4,
				"max_steps": 500,
				"warmup_steps": 25,
				"optim": "paged_adamw_8bit",
				"assistant_only_loss": True,
			}),
		]
	),
	(
		"OpenThoughts (Full Train)", "full_train",
		[
			("Openthoughts-114k", openthoughts.get_prepared_dataset, {
				"lr": 3e-7,
				"batch_size": 4,
				"gradient_steps": 4,
				"max_steps": 500,
				"warmup_steps": 25,
				"optim": "paged_adamw_8bit",
				"assistant_only_loss": True,
			}),
		]
	),
	(
		"Pythoncodes (LoRA)", "lora",
		[
			("Pythoncodes", pythoncodes.get_prepared_dataset, {
				"lr": 3e-6,
				"batch_size": 8,
				"gradient_steps": 8,
				"max_steps": 500,
				"warmup_steps": 25,
				"optim": "paged_adamw_8bit",
				"assistant_only_loss": True,
			}),
		]
	),
	(
		"Pythoncodes (Full Train)", "full_train",
		[
			("Pythoncodes", pythoncodes.get_prepared_dataset, {
				"lr": 3e-7,
				"batch_size": 8,
				"gradient_steps": 8,
				"max_steps": 500,
				"warmup_steps": 25,
				"optim": "paged_adamw_8bit",
				"assistant_only_loss": True,
			}),
		]
	),
]

In [5]:
model, tokenizer = run_experiments(experiments, skip_n=0)
FastLanguageModel.for_inference(model=model)

=== Experiment 1/8: Double Training (LoRA) ===
19:39:11 | INFO | === Запуск этапа 1/2: Openthoughts-114k ===
19:39:11 | INFO | Загрузка базовой модели ./models/qwen3-0.6b...
==((====))==  Unsloth 2026.1.2: Fast Qwen3 patching. Transformers: 4.57.1. vLLM: 0.11.2.
   \\   /|    NVIDIA GeForce RTX 4070 Ti SUPER. Num GPUs = 1. Max memory: 15.992 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
19:39:26 | INFO | Инициализация PEFT/LoRA адаптеров...


Unsloth 2026.1.2 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


19:39:30 | INFO | Составляем датасет для Openthoughts-114k...
19:39:32 | INFO | Берем 8000 примеров из IterableDataset.


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/8000 [00:00<?, ? examples/s]

19:44:16 | INFO | Начинаем обучение на Openthoughts-114k (Оптимизатор: paged_adamw_8bit)...


...

The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 32,000 | Num Epochs = 9,223,372,036,854,775,807 | Total steps = 500
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 8 x 1) = 64
 "-____-"     Trainable parameters = 20,185,088 of 616,235,008 (3.28% trained)
Too many dataloader workers: 4 (max is dataset.num_shards=2). Stopping 2 dataloader workers.
[datasets.iterable_dataset|WARNING]Too many dataloader workers: 4 (max is dataset.num_shards=2). Stopping 2 dataloader workers.


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
10,3.446700
20,3.171800
30,3.206700
40,3.140400
50,3.072800
60,3.106000
70,3.092400
80,3.038500
90,3.059500
100,2.961300


23:12:49 | INFO | Сохраняем объединенную модель в ./models/qwen3-0.6b-sgd-8-exp...
23:12:49 | INFO | Все этапы пайплайна успешно завершены!


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen3ForCausalLM(
      (model): Qwen3Model(
        (embed_tokens): Embedding(151936, 1024, padding_idx=151643)
        (layers): ModuleList(
          (0-27): 28 x Qwen3DecoderLayer(
            (self_attn): Qwen3Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=1024, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1024, out_features=32, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=32, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lor

In [6]:
def quick_sanity_check(model, tokenizer, prompt):
    print(f"🧐 Проверяем промпт: {prompt[:50]}...")

    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

    gen_kwargs = config.SAMPLING_SETTINGS.copy()
    gen_kwargs.update({
        "temperature": 0.6,
        "do_sample": True,
        "max_new_tokens": config.MAX_TOKENS,
        "num_return_sequences": 1,
    })

    if "n" in gen_kwargs:
        gen_kwargs["num_return_sequences"] = gen_kwargs.pop("n")

    for key in ["logprobs", "stop", "ignore_eos", "detokenize"]:
        gen_kwargs.pop(key, None)

    if "max_tokens" in gen_kwargs:
        gen_kwargs["max_new_tokens"] = gen_kwargs.pop("max_tokens")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            pad_token_id=tokenizer.eos_token_id,
            **gen_kwargs
        )

    generated_ids = outputs[0][len(inputs.input_ids[0]):]
    return tokenizer.decode(generated_ids, skip_special_tokens=True)

In [7]:
mbpp_data = mbpp.get_dataset()["test"]
test_sample = mbpp_data[0]

prompt_data = mbpp.build_prompt(test_sample, tokenizer, train=False)
test_prompt = prompt_data["text"]

print("="*40)
print("ВХОДНОЙ ПРОМПТ:")
print(test_prompt)
print("="*40)

# Запускаем проверку
result = quick_sanity_check(model, tokenizer, test_prompt)

print("\n" + "="*40)
print("🤖 ОТВЕТ МОДЕЛИ:")
print("="*40)
print(result)

ВХОДНОЙ ПРОМПТ:
<|im_start|>system
You are an expert Python coding assistant. Given a problem description and function signature, implement the function body so that it passes all tests.<|im_end|>
<|im_start|>user
Problem:
Write a python function to remove first and last occurrence of a given character from the string.

Use the following function signature:
def remove_Occ(s,ch):

Write the full Python function implementation. Do NOT change the function name or arguments. You must analyze the problem in a <think> block first, then provide the solution. The code must be wrapped in markdown block.<|im_end|>
<|im_start|>assistant

🧐 Проверяем промпт: <|im_start|>system
You are an expert Python coding...

🤖 ОТВЕТ МОДЕЛИ:
<think>
Okay, I need to write a Python function called remove_Occ that takes a string s and a character ch. The task is to remove the first and last occurrence of the given character from the string. Let me think about how to approach this.

First, I remember that in Python